## 1. Import and Load Raw Data

In [1]:
import pandas as pd
import numpy as np

SOURCE_FILE = 'data/Central_Superstore.xlsx'

raw = pd.read_excel(SOURCE_FILE)
print('Shape: ', raw.shape)
print(raw.dtypes)
raw.head(2)

Shape:  (2323, 21)
Row ID                    int64
Order ID                 object
Order Date       datetime64[ns]
Ship Date        datetime64[ns]
Ship Mode                object
Customer ID              object
Customer Name            object
Segment                  object
Country                  object
City                     object
State                    object
Postal Code               int64
Region                   object
Product ID               object
Category                 object
Sub-Category             object
Product Name             object
Sales                   float64
Quantity                  int64
Discount                float64
Profit                  float64
dtype: object


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,15,US-2012-118983,2014-11-22,2014-11-26,Standard Class,HP-14815,Harold Pawlan,Home Office,United States,Fort Worth,...,76106,Central,OFF-AP-10002311,Office Supplies,Appliances,Holmes Replacement Filter for HEPA Air Cleaner...,68.810,5,0.8,-123.858
1,16,US-2012-118983,2014-11-22,2014-11-26,Standard Class,HP-14815,Harold Pawlan,Home Office,United States,Fort Worth,...,76106,Central,OFF-BI-10000756,Office Supplies,Binders,Storex DuraTech Recycled Plastic Frosted Binders,2.544,3,0.8,-3.816


## 2. Profile data before cleaning

In [2]:
# NULL COUNT
raw.isnull().sum()

Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64

In [3]:
# DETECT DUPLICATED
raw.duplicated().sum()

np.int64(0)

In [4]:
# PROFILE CUSTOMERS
cust_profile = raw.groupby('Customer ID')[['Customer Name' , 'Segment']].nunique()  # primary key -> Customer ID
print('Customer IDs with > 1 name', (cust_profile['Customer Name'] > 1).sum())
print('Customer IDs with > 1 Segment', (cust_profile['Segment'] > 1).sum())

Customer IDs with > 1 name 0
Customer IDs with > 1 Segment 0


In [5]:
# PROFILE PRODUCTS
pro_profile = raw.groupby('Product ID')[['Product Name' , 'Sub-Category' , 'Category']].nunique()
print('Product IDs with > 1 Product Name', (pro_profile['Product Name'] > 1).sum())   
print('Product IDs with > 1 Sub-Category', (pro_profile['Sub-Category'] > 1).sum())    # primary key -> (Product ID + Product Name)
print('Product IDs with > 1 Category', (pro_profile['Category'] > 1).sum())

Product IDs with > 1 Product Name 16
Product IDs with > 1 Sub-Category 0
Product IDs with > 1 Category 0


In [6]:
order_level = raw.groupby('Order ID')[['Order Date', 'Ship Date', 'Ship Mode', 'Customer ID']].nunique()
print((order_level > 1).sum())

Order Date     0
Ship Date      0
Ship Mode      0
Customer ID    0
dtype: int64


## 3. Cleaning Data

In [7]:
# CHANGE TO SNAKE_CASE
df = raw.copy()
df.columns = ( 
    df.columns
        .str.strip()
        .str.replace(' ','_')
        .str.replace('-','_')
        .str.lower()
)
print(df.columns)

Index(['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode',
       'customer_id', 'customer_name', 'segment', 'country', 'city', 'state',
       'postal_code', 'region', 'product_id', 'category', 'sub_category',
       'product_name', 'sales', 'quantity', 'discount', 'profit'],
      dtype='object')


In [8]:
# TRIM SPACE FOR ALL TEXT COLUMNS
text_columns=['order_id', 'ship_mode','customer_id', 'customer_name', 'segment', 'country', 'city', 'state',
              'region', 'product_id', 'category', 'sub_category', 'product_name']

for c in text_columns :
    df[c] = df[c].astype(str).str.strip()

In [9]:
# CHECK NUMERIC VALUSE
df[df['sales'] <= 0]

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit


In [10]:
df[df['quantity'] <= 0]

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit


In [11]:
df['discount'].unique()

array([0.8 , 0.  , 0.2 , 0.6 , 0.32, 0.3 , 0.5 , 0.4 , 0.1 ])

In [12]:
df.describe()

,row_id,order_date,ship_date,postal_code,sales,quantity,discount,profit
count,2323.000000,2323,2323,2323.000000,2323.000000,2323.000000,2323.000000,2323.000000
mean,5030.547998,2015-04-29 19:22:17.408523520,2015-05-03 20:45:58.501937152,65340.572536,215.772661,3.779595,0.240353,17.092709
min,15.000000,2013-01-03 00:00:00,2013-01-07 00:00:00,46060.000000,0.444000,1.000000,0.000000,-3701.892800
25%,2564.500000,2014-04-29 00:00:00,2014-05-05 00:00:00,55125.000000,14.620000,2.000000,0.000000,-5.664300
50%,5029.000000,2015-07-23 00:00:00,2015-07-27 00:00:00,65203.000000,45.980000,3.000000,0.200000,5.184000
75%,7612.500000,2016-05-09 12:00:00,2016-05-13 00:00:00,77041.000000,200.012000,5.000000,0.300000,22.456200
max,9984.000000,2016-12-30 00:00:00,2017-01-05 00:00:00,79907.000000,17499.950000,14.000000,0.800000,8399.976000
std,2923.638538,NaN,NaN,11427.949045,632.779010,2.160561,0.265433,291.487465


In [13]:
df.dtypes

row_id                    int64
order_id                 object
order_date       datetime64[ns]
ship_date        datetime64[ns]
ship_mode                object
customer_id              object
customer_name            object
segment                  object
country                  object
city                     object
state                    object
postal_code               int64
region                   object
product_id               object
category                 object
sub_category             object
product_name             object
sales                   float64
quantity                  int64
discount                float64
profit                  float64
dtype: object

## 4. Building Dimension Tables
### 4.1 Dim Date

In [14]:
all_dates = pd.concat([df['order_date'],df['ship_date']]).drop_duplicates().sort_values()
print(all_dates)

1809   2013-01-03
188    2013-01-04
1752   2013-01-07
188    2013-01-08
152    2013-01-09
          ...    
163    2016-12-30
237    2017-01-01
1716   2017-01-02
991    2017-01-04
163    2017-01-05
Length: 1058, dtype: datetime64[ns]


In [15]:
dim_date = pd.DataFrame({'full_date' : all_dates})

dim_date['date_key'] = dim_date['full_date'].dt.strftime('%Y%m%d').astype(int)
dim_date['year'] = dim_date['full_date'].dt.year
dim_date['quarter'] = dim_date['full_date'].dt.quarter
dim_date['month'] = dim_date['full_date'].dt.month
dim_date['month_name'] = dim_date['full_date'].dt.month_name()
dim_date['day'] = dim_date['full_date'].dt.day
dim_date['day_name'] = dim_date['full_date'].dt.day_name()
dim_date['week_of_year'] = dim_date['full_date'].dt.isocalendar().week.astype(int)
dim_date['is_weekend'] = dim_date['full_date'].dt.dayofweek >= 5

dim_date = dim_date[['date_key', 'full_date', 'year', 'quarter', 'month', 'month_name', 'day',
                     'day_name', 'week_of_year', 'is_weekend']].reset_index(drop = True)
dim_date.head(3)

,date_key,full_date,year,quarter,month,month_name,day,day_name,week_of_year,is_weekend
0,20130103,2013-01-03,2013,1,1,January,3,Thursday,1,False
1,20130104,2013-01-04,2013,1,1,January,4,Friday,1,False
2,20130107,2013-01-07,2013,1,1,January,7,Monday,2,False


### 4.2 Dim Customer

In [16]:
dim_customer = df[['customer_id', 'customer_name', 'segment']].drop_duplicates().sort_values('customer_id').reset_index(drop = True)
dim_customer.insert(0, 'customer_key', dim_customer.index + 1)

print(dim_customer.shape)
dim_customer.head(3)

(629, 4)


,customer_key,customer_id,customer_name,segment
0,1,AA-10315,Alex Avila,Consumer
1,2,AA-10375,Allen Armold,Consumer
2,3,AA-10480,Andrew Allen,Consumer


### 4.3 Dim Product

In [17]:
dim_product = (
    df[['product_id', 'category', 'sub_category','product_name']]
    .drop_duplicates().sort_values(['product_id','product_name'])
    .reset_index(drop = True)
)
dim_product.insert(0, 'product_key', dim_product.index + 1)

print(dim_product.shape)
dim_product.head(3)

(1326, 5)


,product_key,product_id,category,sub_category,product_name
0,1,FUR-BO-10000112,Furniture,Bookcases,"Bush Birmingham Collection Bookcase, Dark Cherry"
1,2,FUR-BO-10000362,Furniture,Bookcases,Sauder Inglewood Library Bookcases
2,3,FUR-BO-10000468,Furniture,Bookcases,O'Sullivan 2-Shelf Heavy-Duty Bookcases


### 4.4 Dim Location

In [18]:
dim_location = (
    df[['country', 'city', 'state','postal_code', 'region']]
    .drop_duplicates().sort_values(['country', 'city', 'state','postal_code', 'region'])
    .reset_index(drop = True)
)
dim_location.insert(0, 'location_key', dim_location.index + 1)

print(dim_location.shape)
dim_location.head(3)

(195, 6)


,location_key,country,city,state,postal_code,region
0,1,United States,Aberdeen,South Dakota,57401,Central
1,2,United States,Abilene,Texas,79605,Central
2,3,United States,Allen,Texas,75002,Central


### 4.5 Dim Ship_Mode

In [19]:
dim_ship_mode = df[['ship_mode']].drop_duplicates().sort_values('ship_mode').reset_index(drop = True)
dim_ship_mode.insert(0, 'ship_mode_key', dim_ship_mode.index + 1)

print(dim_ship_mode.shape)
dim_ship_mode.head(3)

(4, 2)


,ship_mode_key,ship_mode
0,1,First Class
1,2,Same Day
2,3,Second Class


## 5. Building Fact Table


In [20]:
fact_order  = df.merge(dim_customer, on=["customer_id","customer_name","segment"], how="left") \
                .merge(dim_product, on=["product_id","product_name","category","sub_category"], how="left") \
                .merge(dim_location, on=["country","region","state","city","postal_code"], how="left") \
                .merge(dim_ship_mode, on="ship_mode", how="left") \
                .merge(dim_date[["date_key","full_date"]].rename(columns={"date_key":"order_date_key","full_date":"order_date"}), 
                       on = "order_date", how="left") \
                .merge(dim_date[["date_key","full_date"]].rename(columns={"date_key":"ship_date_key","full_date":"ship_date"}), 
                       on = "ship_date", how="left")

fact_order.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2323 entries, 0 to 2322
Data columns (total 27 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   row_id          2323 non-null   int64         
 1   order_id        2323 non-null   object        
 2   order_date      2323 non-null   datetime64[ns]
 3   ship_date       2323 non-null   datetime64[ns]
 4   ship_mode       2323 non-null   object        
 5   customer_id     2323 non-null   object        
 6   customer_name   2323 non-null   object        
 7   segment         2323 non-null   object        
 8   country         2323 non-null   object        
 9   city            2323 non-null   object        
 10  state           2323 non-null   object        
 11  postal_code     2323 non-null   int64         
 12  region          2323 non-null   object        
 13  product_id      2323 non-null   object        
 14  category        2323 non-null   object        
 15  sub_

In [21]:
fact_order = fact_order[['row_id','order_id', 'order_date_key', 'ship_date_key', 'ship_mode_key','customer_key',
                         'location_key', 'product_key', 'sales', 'quantity', 'discount',
                         'profit']].rename(columns = {'row_id':'sales_key'}).sort_values('sales_key').reset_index(drop = True)

fact_order.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2323 entries, 0 to 2322
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sales_key       2323 non-null   int64  
 1   order_id        2323 non-null   object 
 2   order_date_key  2323 non-null   int64  
 3   ship_date_key   2323 non-null   int64  
 4   ship_mode_key   2323 non-null   int64  
 5   customer_key    2323 non-null   int64  
 6   location_key    2323 non-null   int64  
 7   product_key     2323 non-null   int64  
 8   sales           2323 non-null   float64
 9   quantity        2323 non-null   int64  
 10  discount        2323 non-null   float64
 11  profit          2323 non-null   float64
dtypes: float64(3), int64(8), object(1)
memory usage: 217.9+ KB


## 6. Connecting to SQL Server

In [22]:
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings("ignore", category=Warning)

SQL_SERVER = 'localhost\\MSSQLSERVER2026'
SQL_DATABASE = 'CenterSupertoreDW'
SQL_DRIVER = 'ODBC Driver 17 for SQL Server'

#windows auth
conn_str = (
    f"mssql+pyodbc://@{SQL_SERVER}/{SQL_DATABASE}"
    f"?driver={SQL_DRIVER.replace(' ', '+')}&trusted_connection=yes"
)
engine = create_engine(conn_str, fast_executemany=True)
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT @@VERSION"))
        print(result.fetchone()[0])
except Exception as e:
    print("Connection Failed", e)

Microsoft SQL Server 2025 (RTM) - 17.0.1000.7 (X64) 
	Oct 21 2025 12:05:57 
	Copyright (C) 2025 Microsoft Corporation
	Standard Developer Edition (64-bit) on Windows 10 Pro 10.0 <X64> (Build 19045: )



## 7. Create the schema and tables

In [24]:
ddl_statements = [
    """
    CREATE TABLE dbo.dim_date (
        date_key INT PRIMARY KEY,
        full_date DATE NOT NULL,
        year SMALLINT NOT NULL,
        quarter TINYINT NOT NULL,
        month SMALLINT NOT NULL,
        month_name VARCHAR(15) NOT NULL,
        day TINYINT NOT NULL,
        day_name VARCHAR(15) NOT NULL,
        week_of_year TINYINT NOT NULL,
        is_weekend BIT NOT NULL
    );
    """
    ,
    """
    CREATE TABLE dbo.dim_customer (
        customer_key INT PRIMARY KEY,
        customer_id VARCHAR(20) NOT NULL,
        customer_name VARCHAR(100) NOT NULL,
        segment VARCHAR(20) NOT NULL
    );
    """
    ,
    """
    CREATE TABLE dbo.dim_product (
        product_key INT PRIMARY KEY,
        product_id VARCHAR(20) NOT NULL,
        product_name VARCHAR(250) NOT NULL,
        category VARCHAR(20) NOT NULL,
        sub_category VARCHAR(20) NOT NULL
    );
    """
    ,
    """
    CREATE TABLE dbo.dim_location (
        location_key INT PRIMARY KEY,
        country VARCHAR(50) NOT NULL,
        region VARCHAR(50) NOT NULL,
        state VARCHAR(50) NOT NULL,
        city VARCHAR(50) NOT NULL,
        postal_code INT NOT NULL
    );
    """
    ,
    """
    CREATE TABLE dbo.dim_ship_mode (
        ship_mode_key INT PRIMARY KEY,
        ship_mode VARCHAR(30) NOT NULL
    );
    """
    ,
    """
    CREATE TABLE dbo.fact_order (
        sales_key INT PRIMARY KEY,
        order_id VARCHAR(20) NOT NULL,
        order_date_key INT NOT NULL,
        ship_date_key INT NOT NULL,
        ship_mode_key INT NOT NULL,
        customer_key INT NOT NULL,
        location_key INT NOT NULL,
        product_key INT NOT NULL,
        sales DECIMAL(12,4) NOT NULL,
        quantity INT NOT NULL,
        discount DECIMAL(5,4) NOT NULL,
        profit DECIMAL(12,4) NOT NULL,
        CONSTRAINT fk_fact_order_date FOREIGN KEY (order_date_key) REFERENCES dbo.dim_date(date_key),
        CONSTRAINT fk_fact_ship_date FOREIGN KEY (ship_date_key) REFERENCES dbo.dim_date(date_key),
        CONSTRAINT fk_fact_ship_mode FOREIGN KEY (ship_mode_key) REFERENCES dbo.dim_ship_mode(ship_mode_key),
        CONSTRAINT fk_fact_customer FOREIGN KEY (customer_key) REFERENCES dbo.dim_customer(customer_key),
        CONSTRAINT fk_fact_product FOREIGN KEY (product_key) REFERENCES dbo.dim_product(product_key),
        CONSTRAINT fk_fact_location FOREIGN KEY (location_key) REFERENCES dbo.dim_location(location_key),
    );
    """
]

with engine.begin() as conn:
    for stmt in ddl_statements:
        conn.execute(text(stmt))

print("ALL TABLES CREATED")

ALL TABLES CREATED


## 8. Load Data into SQL Server Tables

In [25]:
load_order = [
    ("dim_date", dim_date),
    ("dim_ship_mode", dim_ship_mode),
    ("dim_customer", dim_customer),
    ("dim_location", dim_location),
    ("dim_product", dim_product),
    ("fact_order", fact_order)
]

for table_name, df in load_order:
    df.to_sql(table_name, con=engine, schema="dbo", if_exists="append",
             index=False, chunksize=1000)
    print(f"Loaded {len(df)} rows --into--> dbo.{table_name}")

Loaded 1058 rows --into--> dbo.dim_date
Loaded 4 rows --into--> dbo.dim_ship_mode
Loaded 629 rows --into--> dbo.dim_customer
Loaded 195 rows --into--> dbo.dim_location
Loaded 1326 rows --into--> dbo.dim_product
Loaded 2323 rows --into--> dbo.fact_order


## 9. Verify Loaded Data

In [26]:
query = """
    SELECT 
         f.order_id
        ,d.full_date AS order_date
        ,sm.ship_mode
        ,c.customer_name
        ,p.product_name
        ,l.country
        ,l.city
        ,l.state
        ,f.sales
        ,f.quantity
        ,f.discount
        ,f.profit
    FROM dbo.fact_order f
    JOIN dbo.dim_date d
        ON f.order_date_key = d.date_key
    JOIN dbo.dim_customer c
        ON f.customer_key = c.customer_key
    JOIN dbo.dim_product p
        ON f.product_key = p.product_key
    JOIN dbo.dim_location l
        ON f.location_key = l.location_key
    JOIN dbo.dim_ship_mode sm
        ON f.ship_mode_key = sm.ship_mode_key
"""

new_df = pd.read_sql(query, engine)
new_df.head(3)

,order_id,order_date,ship_mode,customer_name,product_name,country,city,state,sales,quantity,discount,profit
0,US-2012-118983,2014-11-22,Standard Class,Harold Pawlan,Holmes Replacement Filter for HEPA Air Cleaner...,United States,Fort Worth,Texas,68.810,5,0.8,-123.8580
1,US-2012-118983,2014-11-22,Standard Class,Harold Pawlan,Storex DuraTech Recycled Plastic Frosted Binders,United States,Fort Worth,Texas,2.544,3,0.8,-3.8160
2,CA-2011-105893,2013-11-11,Standard Class,Pete Kriz,"Stur-D-Stor Shelving, Vertical 5-Shelf: 72""H x...",United States,Madison,Wisconsin,665.880,6,0.0,13.3176
